# POS Tagging con NLTK

In [1]:
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
import nltk
from nltk.corpus import cess_esp
import time
#nltk.download('cess_esp')

df = pd.read_csv('../data/processed/resenas_clean.csv', sep=';')
df = df.drop(columns=['total_reviews'])
df.sample(2)

,business_name,review_rating,review_text,datetime_utc
1119,Museo de los Niños Costa Rica,5,Por el mes de enero la entrada está en ₡1.000 ...,2023/01/18 02:06:29
1376,Museo de los Niños Costa Rica,5,Fue muy hermoso volver después de muchos años....,2023/02/19 14:48:23


## Utilizando el etiquetador por defecto (no funciona bien en español)

In [2]:
oraciones_ejemplo = df.sample(5, random_state=3)['review_text']

tiempos = []
for i in range(10):
    inicio = time.perf_counter()
    
    # ---
    for sent in oraciones_ejemplo:
        print(f"Oración: {sent}")
        tokens = word_tokenize(sent)
        pos_tags = pos_tag(tokens)
        for word, tag in pos_tags:
            print(f"  {word:15} → {tag}")
        print()
    # ----
    
    fin = time.perf_counter()
    tiempo_ejecucion = fin - inicio
    tiempos.append(tiempo_ejecucion)
    
promedio = sum(tiempos) / len(tiempos)

print("--- RESULTADOS FINALES ---")
print(f"Tiempos individuales: {[f'{t:.6f} s' for t in tiempos]}")
print(f"Tiempo promedio de ejecución: {promedio:.6f} segundos")

Oración: Esto es en Colombia Me falto llega a Costa Rica
  Esto            → NNP
  es              → NN
  en              → NN
  Colombia        → NNP
  Me              → NNP
  falto           → VBZ
  llega           → VBZ
  a               → DT
  Costa           → NNP
  Rica            → NNP

Oración: Bonita temática para niñas, niñas y adultos
  Bonita          → NNP
  temática        → NN
  para            → NN
  niñas           → NN
  ,               → ,
  niñas           → JJ
  y               → NN
  adultos         → NN

Oración: El lugar es muy bonito, el tobogán que se parece al paquare es un tanto peligroso. Yo y unos amigos al caer a la piscina con la balsa, esta provocó que un amigo terminara en el agua. Las termales podrían tener alguna que otra piscina más caliente. Pero definitivamente es un lugar para visitar. Y punto muy importante, los servicios sanitarios estaban MUY limpios.
  El              → NNP
  lugar           → NN
  es              → NN
  muy             → NN


## Utilizando un etiquetador para español

### Tiempo de ejecución

In [3]:
training_sentences = cess_esp.tagged_sents()

default_tagger = nltk.DefaultTagger('nc0s000')
spanish_tagger = nltk.UnigramTagger(training_sentences, backoff=default_tagger)

tiempos = []
for i in range(10):
    inicio = time.perf_counter()
    
    # ---
    for sent in oraciones_ejemplo:
        print(f"Oración: {sent}")
        tokens = word_tokenize(sent)
        pos_tags = spanish_tagger.tag(tokens)
        for word, tag in pos_tags:
            print(f"  {word:15} → {tag}")
        print()
    # ---
    
    fin = time.perf_counter()
    tiempo_ejecucion = fin - inicio
    tiempos.append(tiempo_ejecucion)
    
promedio = sum(tiempos) / len(tiempos)

print("--- RESULTADOS FINALES ---")
print(f"Tiempos individuales: {[f'{t:.6f} s' for t in tiempos]}")
print(f"Tiempo promedio de ejecución: {promedio:.6f} segundos")

Oración: Esto es en Colombia Me falto llega a Costa Rica
  Esto            → pd0ns000
  es              → vsip3s0
  en              → sps00
  Colombia        → np0000l
  Me              → pp1cs000
  falto           → nc0s000
  llega           → vmip3s0
  a               → sps00
  Costa           → nc0s000
  Rica            → nc0s000

Oración: Bonita temática para niñas, niñas y adultos
  Bonita          → nc0s000
  temática        → aq0fs0
  para            → sps00
  niñas           → ncfp000
  ,               → Fc
  niñas           → ncfp000
  y               → cc
  adultos         → ncmp000

Oración: El lugar es muy bonito, el tobogán que se parece al paquare es un tanto peligroso. Yo y unos amigos al caer a la piscina con la balsa, esta provocó que un amigo terminara en el agua. Las termales podrían tener alguna que otra piscina más caliente. Pero definitivamente es un lugar para visitar. Y punto muy importante, los servicios sanitarios estaban MUY limpios.
  El              → da0ms

### Precisión

In [4]:
oraciones_EAGLE = [
    [
        ("La", "da0fs0"), ("comida", "ncfs000"), ("estaba", "vmii3s0"), ("demasiado", "rg"), 
        ("fría", "aq0fs0"), (",", "Fc"), ("aunque", "cs"), ("el", "da0ms0"), 
        ("servicio", "ncms000"), ("fue", "vmsis30"), ("muy", "rg"), ("atento", "aq0ms0"), (".", "Fp")
    ],
    [
        ("Habiendo", "vmg0000"), ("esperado", "vmp0000"), ("horas", "ncfp000"), (",", "Fc"), 
        ("el", "da0ms0"), ("guía", "ncms000"), ("turístico", "aq0ms0"), ("parecía", "vmii3s0"), 
        ("desorientado", "aq0ms0"), ("buscando", "vmg0000"), ("el", "da0ms0"), ("museo", "ncms000"), (".", "Fp")
    ],
    [
        ("Visité", "vmis1s0"), ("el", "da0ms0"), ("banco", "ncms000"), ("donde", "pr00000"), 
        ("se", "p0000000"), ("cambia", "vmip3s0"), ("moneda", "ncfs000"), (",", "Fc"), 
        ("porque", "cs"), ("necesitaba", "vmii3s0"), ("efectivo", "ncms000"), ("urgente", "aq0cs0"), (".", "Fp")
    ]
]

total_tokens = 0
errores = 0

print(f"{'PALABRA':<15} | {'REAL (EAGLES)':<15} | {'NLTK (EAGLES)':<15}")
print("-" * 55)

for oracion_data in oraciones_EAGLE:
    palabras = [t[0] for t in oracion_data]
    resultado_nltk = spanish_tagger.tag(palabras)
    
    for i, token_data in enumerate(oracion_data):
        total_tokens += 1
        palabra_real, tag_real = token_data
        _, tag_nltk = resultado_nltk[i]
        
        if tag_real != tag_nltk:
            errores += 1
            print(f"{palabra_real:<15} | {tag_real:<15} | {str(tag_nltk):<15}")

precision = ((total_tokens - errores) / total_tokens) * 100
print("-" * 55)
print(f"Precisión total con NLTK: {precision:.2f}%")

PALABRA         | REAL (EAGLES)   | NLTK (EAGLES)  
-------------------------------------------------------
fue             | vmsis30         | vsis3s0        
Habiendo        | vmg0000         | nc0s000        
esperado        | vmp0000         | aq0msp         
guía            | ncms000         | ncfs000        
desorientado    | aq0ms0          | aq0msp         
Visité          | vmis1s0         | nc0s000        
donde           | pr00000         | pr000000       
se              | p0000000        | p0300000       
efectivo        | ncms000         | aq0ms0         
-------------------------------------------------------
Precisión total con NLTK: 76.92%


### Sistema EAGLES

Cada letra y posición tiene un significado específico. Por ejemplo

| Palabra | Etiqueta POS | Descripción |
| :--- | :--- | :--- |
| **"Esto"** | `pd0ns000` | Pronombre demostrativo, neutro y singular. |
| **"es"** | `vsip3s0` | Verbo semicopulativo/auxiliar, en presente de indicativo, tercera persona del singular. |
| **"Colombia"** | `np0000l` | Nombre propio de lugar. |